# 04 - Ratings per film and per year for the 142 nominees

Inputs: `data/raw/ml-32m/ratings.csv` and `data/processed/best_picture_nominees_1996_2017.csv` (built by `src/build_nominees_table.py`, see notebook 03).

Status: work in progress. The analysis window and the minimum number of ratings are being discussed again.

Window: from the ceremony year to the ceremony year + 5. All years are computed in UTC.

In [1]:
import duckdb

con = duckdb.connect()
con.sql("SET enable_progress_bar = false")  # no progress bar in the saved outputs

## A. A worked example: Titanic (movieId 1721) and platform activity

In [2]:
# Step 7: Titanic (movieId 1721), number of ratings and average rating per year (UTC)
con.sql("SELECT year(to_timestamp(timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings, ROUND(AVG(rating), 2) AS avg_rating FROM '../data/raw/ml-32m/ratings.csv' WHERE movieId = 1721 GROUP BY rating_year ORDER BY rating_year").show(max_rows=40)

┌─────────────┬────────────┬────────────┐
│ rating_year │ nb_ratings │ avg_rating │
│    int64    │   int64    │   double   │
├─────────────┼────────────┼────────────┤
│        1997 │         39 │       4.31 │
│        1998 │       1542 │       4.11 │
│        1999 │       3525 │       3.32 │
│        2000 │       2929 │       3.58 │
│        2001 │       1826 │       3.51 │
│        2002 │       1378 │       3.28 │
│        2003 │       1189 │       3.27 │
│        2004 │       1986 │       3.15 │
│        2005 │       2805 │       3.06 │
│        2006 │       2211 │        3.2 │
│        2007 │       2316 │       3.13 │
│        2008 │       2556 │       3.22 │
│        2009 │       1669 │       3.28 │
│        2010 │       1629 │       3.31 │
│        2011 │       1287 │       3.41 │
│        2012 │       1006 │       3.49 │
│        2013 │        665 │       3.51 │
│        2014 │        528 │        3.4 │
│        2015 │       1878 │       3.44 │
│        2016 │       2881 │      

In [3]:
# Step 8: date and time (UTC) of the very first rating of Titanic (movieId 1721)
con.sql("SELECT to_timestamp(MIN(timestamp)) AT TIME ZONE 'UTC' AS first_rating_utc FROM '../data/raw/ml-32m/ratings.csv' WHERE movieId = 1721")

┌─────────────────────┐
│  first_rating_utc   │
│      timestamp      │
├─────────────────────┤
│ 1997-12-19 19:57:55 │
└─────────────────────┘

In [4]:
# Step 9: total number of ratings per year in the whole dataset (all movies), UTC
con.sql("SELECT year(to_timestamp(timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings FROM '../data/raw/ml-32m/ratings.csv' GROUP BY rating_year ORDER BY rating_year").show(max_rows=40)

┌─────────────┬────────────┐
│ rating_year │ nb_ratings │
│    int64    │   int64    │
├─────────────┼────────────┤
│        1995 │          4 │
│        1996 │    1571368 │
│        1997 │     685388 │
│        1998 │     301691 │
│        1999 │    1174629 │
│        2000 │    1912322 │
│        2001 │    1160098 │
│        2002 │     849762 │
│        2003 │    1011000 │
│        2004 │    1139068 │
│        2005 │    1752573 │
│        2006 │    1141704 │
│        2007 │    1023813 │
│        2008 │    1117071 │
│        2009 │     890696 │
│        2010 │     860073 │
│        2011 │     729301 │
│        2012 │     695493 │
│        2013 │     563301 │
│        2014 │     518537 │
│        2015 │    1743866 │
│        2016 │    1918739 │
│        2017 │    1827953 │
│        2018 │    1391057 │
│        2019 │    1385467 │
│        2020 │    1688159 │
│        2021 │    1239912 │
│        2022 │     913105 │
│        2023 │     794054 │
├─────────────┴────────────┤
│ 29 rows     

## B. The table film x year for all nominees

In [5]:
# Step 11: for each of the 142 nominees and each rating year (UTC): number of ratings and average rating
con.sql("SELECT n.ceremony_year, n.title, n.is_winner, n.movieId, year(to_timestamp(r.timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings, ROUND(AVG(r.rating), 2) AS avg_rating FROM '../data/processed/best_picture_nominees_1996_2017.csv' n JOIN '../data/raw/ml-32m/ratings.csv' r ON n.movieId = r.movieId GROUP BY n.ceremony_year, n.title, n.is_winner, n.movieId, rating_year ORDER BY n.ceremony_year, n.title, rating_year").show(max_rows=30)


┌───────────────┬───────────────────────┬───────────┬─────────┬─────────────┬────────────┬────────────┐
│ ceremony_year │         title         │ is_winner │ movieId │ rating_year │ nb_ratings │ avg_rating │
│     int64     │        varchar        │  boolean  │  int64  │    int64    │   int64    │   double   │
├───────────────┼───────────────────────┼───────────┼─────────┼─────────────┼────────────┼────────────┤
│          1996 │ Apollo 13             │ false     │     150 │        1996 │      19102 │        4.0 │
│          1996 │ Apollo 13             │ false     │     150 │        1997 │       1624 │       3.98 │
│          1996 │ Apollo 13             │ false     │     150 │        1998 │        990 │       4.04 │
│          1996 │ Apollo 13             │ false     │     150 │        1999 │       2024 │        4.0 │
│          1996 │ Apollo 13             │ false     │     150 │        2000 │       2397 │       4.02 │
│          1996 │ Apollo 13             │ false     │     150 │ 

In [6]:
# Step 12: create a reusable view with the number of ratings and average rating per nominee and per rating year (UTC)
con.sql("CREATE VIEW film_year AS SELECT n.ceremony_year, n.title, n.is_winner, n.movieId, year(to_timestamp(r.timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings, ROUND(AVG(r.rating), 2) AS avg_rating FROM '../data/processed/best_picture_nominees_1996_2017.csv' n JOIN '../data/raw/ml-32m/ratings.csv' r ON n.movieId = r.movieId GROUP BY n.ceremony_year, n.title, n.is_winner, n.movieId, rating_year")

## C. Apply the window

The window runs from the ceremony year to the ceremony year + 5 (6 calendar years), the same for every film.
We apply it first; the minimum number of ratings will be discussed afterwards, on the rows that are inside the window.

In [7]:
# Step 1: keep only the rows inside the window, and add the number of years since the ceremony (0 to 5)
con.sql("CREATE VIEW film_year_window AS SELECT *, rating_year - ceremony_year AS years_since_ceremony FROM film_year WHERE rating_year BETWEEN ceremony_year AND ceremony_year + 5")

In [8]:
# Step 2: check the window: how many rows for each year since the ceremony (we expect 142 films each time)
con.sql("SELECT years_since_ceremony, COUNT(*) AS n_rows FROM film_year_window GROUP BY years_since_ceremony ORDER BY years_since_ceremony")

┌──────────────────────┬────────┐
│ years_since_ceremony │ n_rows │
│        int64         │ int64  │
├──────────────────────┼────────┤
│                    0 │    142 │
│                    1 │    142 │
│                    2 │    142 │
│                    3 │    142 │
│                    4 │    142 │
│                    5 │    142 │
└──────────────────────┴────────┘

In [9]:
# Step 3: total number of rows in the window, and number of different films
con.sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT movieId) AS n_films FROM film_year_window")

┌────────┬─────────┐
│ n_rows │ n_films │
│ int64  │  int64  │
├────────┼─────────┤
│    852 │     142 │
└────────┴─────────┘

## D. How reliable are the yearly averages? Testing a minimum number of ratings

A yearly average built on few ratings is unreliable. To choose a minimum number of ratings we look at four things, in this order, inside the window only:
1. how the number of ratings per row is distributed,
2. how many rows would fall below several candidate minimums,
3. which films and years those rows are,
4. what precision (error of an average) each minimum gives.

Nothing is removed in this section.

In [10]:
# Step 4: distribution of the number of ratings per row (smallest, quartiles, median, largest)
con.sql("SELECT MIN(nb_ratings) AS smallest, QUANTILE_CONT(nb_ratings, 0.25) AS q25, MEDIAN(nb_ratings) AS median, QUANTILE_CONT(nb_ratings, 0.75) AS q75, MAX(nb_ratings) AS largest FROM film_year_window")

┌──────────┬────────┬────────┬─────────┬─────────┐
│ smallest │  q25   │ median │   q75   │ largest │
│  int64   │ double │ double │ double  │  int64  │
├──────────┼────────┼────────┼─────────┼─────────┤
│       29 │  420.0 │  931.5 │ 1689.25 │   19102 │
└──────────┴────────┴────────┴─────────┴─────────┘

In [11]:
# Step 5: number of rows below several candidate minimums (nothing is removed)
con.sql("SELECT COUNT(*) AS total_rows, SUM(CASE WHEN nb_ratings < 10 THEN 1 ELSE 0 END) AS below_10, SUM(CASE WHEN nb_ratings < 30 THEN 1 ELSE 0 END) AS below_30, SUM(CASE WHEN nb_ratings < 50 THEN 1 ELSE 0 END) AS below_50, SUM(CASE WHEN nb_ratings < 100 THEN 1 ELSE 0 END) AS below_100 FROM film_year_window")

┌────────────┬──────────┬──────────┬──────────┬───────────┐
│ total_rows │ below_10 │ below_30 │ below_50 │ below_100 │
│   int64    │  int128  │  int128  │  int128  │  int128   │
├────────────┼──────────┼──────────┼──────────┼───────────┤
│        852 │        0 │        2 │        6 │        28 │
└────────────┴──────────┴──────────┴──────────┴───────────┘

In [12]:
# Step 6: the rows with fewer than 100 ratings, smallest first (which film, which year)
con.sql("SELECT ceremony_year, title, rating_year, years_since_ceremony, nb_ratings FROM film_year_window WHERE nb_ratings < 100 ORDER BY nb_ratings, ceremony_year, title").show(max_rows=40)

┌───────────────┬─────────────────────────────────────────────────┬─────────────┬──────────────────────┬────────────┐
│ ceremony_year │                      title                      │ rating_year │ years_since_ceremony │ nb_ratings │
│     int64     │                     varchar                     │    int64    │        int64         │   int64    │
├───────────────┼─────────────────────────────────────────────────┼─────────────┼──────────────────────┼────────────┤
│          2012 │ Extremely Loud & Incredibly Close               │        2014 │                    2 │         29 │
│          2012 │ War Horse                                       │        2014 │                    2 │         29 │
│          2011 │ The Kids Are All Right                          │        2014 │                    3 │         38 │
│          2017 │ Fences                                          │        2022 │                    5 │         40 │
│          2010 │ Precious: Based on the Novel 'Push' by

In [13]:
# Step 7: standard deviation of all the ratings given to the 142 nominees (used to compute the error of an average)
con.sql("SELECT ROUND(STDDEV(r.rating), 2) AS std_rating FROM '../data/processed/best_picture_nominees_1996_2017.csv' n JOIN '../data/raw/ml-32m/ratings.csv' r ON n.movieId = r.movieId")

┌────────────┐
│ std_rating │
│   double   │
├────────────┤
│       0.94 │
└────────────┘